## Data analysis

In [1]:
import sys
import os

# Add the parent directory (simcode) to sys.path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))

In [2]:
import pandas as pd
import numpy as np
import json
from ecv_rank import *
from src.solvers.weighted_network import ASE
from src.methods import * 
import matplotlib.pyplot as plt
import requests

In [3]:
edges = pd.read_csv('wikipedia/chameleon/musae_chameleon_edges.csv', header=0)

In [4]:
edges['id1'] = edges['id1'].astype(int)
edges['id2'] = edges['id2'].astype(int)

In [5]:
adj_matrix = np.zeros((edges['id1'].max() + 1, edges['id2'].max() + 1))
adj_matrix[edges['id1'], edges['id2']] = 1
adj_matrix[edges['id2'], edges['id1']] = 1  # Assuming undirected graph
adj_matrix[np.diag_indices_from(adj_matrix)] = 0  # Remove self-loops

In [6]:
with open('wikipedia/chameleon/musae_chameleon_features.json', 'r') as f:
    covariates = json.load(f)

In [7]:
m = -1
for x in covariates:
    for y in covariates[x]:
        if y > m:
            m = y

In [8]:
cov_matrix = np.zeros(shape=(adj_matrix.shape[0], m+1))

for x in covariates:
    for y in covariates[x]:
        cov_matrix[int(x), int(y)] = y

In [9]:
outcome = pd.read_csv('wikipedia/chameleon/musae_chameleon_target.csv', header=0)

In [10]:
cov_matrix_num_nouns = np.hstack([cov_matrix, cov_matrix.sum(axis=1).reshape(-1, 1)])
cov_matrix_log_num_nouns = np.hstack([cov_matrix, np.log1p(cov_matrix.sum(axis=1).reshape(-1, 1))])
cov_matrix_with_outcome = np.hstack([cov_matrix_num_nouns, outcome['target'].to_numpy().reshape(-1, 1)])
cov_matrix_log_outcome = np.hstack([cov_matrix_log_num_nouns, np.log1p(outcome['target'].to_numpy().reshape(-1, 1))])
cov_only_outcome = outcome['target'].to_numpy().reshape(-1, 1)
cov_outcome_nouns = np.hstack([cov_matrix.sum(axis=1).reshape(-1, 1), outcome['target'].to_numpy().reshape(-1, 1)])
cov_outcome_nouns_log = np.hstack([np.log1p(cov_matrix.sum(axis=1).reshape(-1, 1)), np.log1p(outcome['target'].to_numpy().reshape(-1, 1))])

In [11]:
data_nouns = {'A': adj_matrix, 'Y': cov_matrix_num_nouns}  
data_log_nouns = {'A': adj_matrix, 'Y': cov_matrix_log_num_nouns}
data_with_outcome = {'A': adj_matrix, 'Y': cov_matrix_with_outcome}
data_log_outcome = {'A': adj_matrix, 'Y': cov_matrix_log_outcome}
data_only_outcome = {'A': adj_matrix, 'Y': cov_only_outcome}
data_outcome_nouns = {'A': adj_matrix, 'Y': cov_outcome_nouns}
data_outcome_nouns_log = {'A': adj_matrix, 'Y': cov_outcome_nouns_log}

In [ ]:
# result = ecv_rank(
#     adj_matrix,
#     max_k=100,
#     B=10,
#     holdout_p=0.1,
#     weighted=False,
#     mode="undirected",
#     random_state=1,
# )

# print("SSE selected rank:", result.sse_rank)
# print("AUC selected rank:", result.auc_rank)

SSE selected rank: 55
AUC selected rank: 58


In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler(with_std=False)  # Set with_std=False to avoid scaling by standard deviation
cov_matrix_scaled = scaler.fit_transform(cov_matrix)
pca = PCA(n_components=0.9)
pca.fit(cov_matrix_scaled)
cov_matrix_reduced = pca.transform(cov_matrix_scaled)
cov_matrix_reduced

In [ ]:
cov_matrix.shape

(2277, 3132)

In [13]:
rv_test = RVTest(k=55, 
                 solver=ASE, 
                 random_state=1, 
                 verbose=True,
                 permutation_type='covariate',
                 npermutations=1000)

rv_test.fit(data_outcome_nouns_log)

rv_test.get_estimated()['p-value'], rv_test.get_estimated()['test_stat'], rv_test.get_estimated()['reject_null']

Permutations:   0%|          | 0/1000 [00:00<?, ?permutation/s]

(0.000999000999000999, 0.07151137973194928, True)

In [14]:
test_ac = MultivariateACTest(k=55, 
                             solver=ASE, 
                             M=1, 
                             random_state=1, 
                             verbose=True, 
                             permutation_type='covariate', 
                             n_jobs=4, 
                             npermutations=1000,
                             use_permutation_coeff=True
                          )

test_ac.fit(data_outcome_nouns_log)

test_ac.get_estimated()['p-value'], test_ac.get_estimated()['test_stat'], test_ac.get_estimated()['reject_null']

Permutations:   0%|          | 0/1000 [00:00<?, ?permutation/s]

(0.000999000999000999, 0.6717645225664841, True)

In [15]:
test_ac_adaptive = MultivariateACTest(k=55, 
                                      solver=ASE, 
                                      adaptive_m=True, 
                                      random_state=1, 
                                      verbose=True, 
                                      permutation_type='covariate', 
                                      n_jobs=-1,
                                      use_permutation_coeff=True,
                                      npermutations=300)

test_ac_adaptive.fit(data_outcome_nouns)

test_ac_adaptive.get_estimated()['p-value'], test_ac_adaptive.get_estimated()['test_stat'], test_ac_adaptive.get_estimated()['reject_null']

Permutations:   0%|          | 0/300 [00:00<?, ?permutation/s]

(0.0033222591362126247, 16.640081020391495, True)

In [16]:
rv_test = RVTest(k=55, 
                 solver=ASE, 
                 random_state=1, 
                 verbose=True,
                 permutation_type='covariate',
                 npermutations=1000)

rv_test.fit(data_outcome_nouns)

rv_test.get_estimated()['p-value'], rv_test.get_estimated()['test_stat'], rv_test.get_estimated()['reject_null']

Permutations:   0%|          | 0/1000 [00:00<?, ?permutation/s]

(0.000999000999000999, 0.041090002222194014, True)

In [17]:
test_ac_adaptive = MultivariateACTest(k=55, 
                                      solver=ASE, 
                                      adaptive_m=True, 
                                      random_state=1, 
                                      verbose=True, 
                                      permutation_type='covariate', 
                                      n_jobs=-1,
                                      use_permutation_coeff=True,
                                      npermutations=300)

test_ac_adaptive.fit(data_outcome_nouns)

test_ac_adaptive.get_estimated()['p-value'], test_ac_adaptive.get_estimated()['test_stat'], test_ac_adaptive.get_estimated()['reject_null']

Permutations:   0%|          | 0/300 [00:00<?, ?permutation/s]

(0.0033222591362126247, 16.602740045158217, True)

In [18]:
rv_test = RVTest(k=55, 
                 solver=ASE, 
                 random_state=1, 
                 verbose=True,
                 permutation_type='covariate',
                 npermutations=1000)

rv_test.fit(data_only_outcome)

rv_test.get_estimated()['p-value'], rv_test.get_estimated()['test_stat'], rv_test.get_estimated()['reject_null']

Permutations:   0%|          | 0/1000 [00:00<?, ?permutation/s]

(0.000999000999000999, 0.017659098982104597, True)

In [19]:
test_ac_adaptive = MultivariateACTest(k=55, 
                                      solver=ASE, 
                                      adaptive_m=True, 
                                      random_state=1, 
                                      verbose=True, 
                                      permutation_type='covariate', 
                                      n_jobs=-1,
                                      use_permutation_coeff=False,
                                      npermutations=300)

test_ac_adaptive.fit(data_only_outcome)

test_ac_adaptive.get_estimated()['p-value'], test_ac_adaptive.get_estimated()['test_stat'], test_ac_adaptive.get_estimated()['reject_null']

Permutations:   0%|          | 0/300 [00:00<?, ?permutation/s]

(0.0033222591362126247, 16.12528735184164, True)

In [20]:
rv_test = RVTest(k=55, 
                 solver=ASE, 
                 random_state=1, 
                 verbose=True,
                 permutation_type='covariate',
                 npermutations=1000)

rv_test.fit(data_with_outcome)

rv_test.get_estimated()['p-value'], rv_test.get_estimated()['test_stat'], rv_test.get_estimated()['reject_null']

Permutations:   0%|          | 0/1000 [00:00<?, ?permutation/s]

(0.000999000999000999, 0.04200424840475305, True)

In [21]:
test_ac_adaptive = MultivariateACTest(k=55, 
                                      solver=ASE, 
                                      M=1,
                                      random_state=1, 
                                      verbose=True, 
                                      permutation_type='covariate', 
                                      n_jobs=-1,
                                      use_permutation_coeff=False,
                                      npermutations=50)

test_ac_adaptive.fit(data_with_outcome)

test_ac_adaptive.get_estimated()['p-value'], test_ac_adaptive.get_estimated()['test_stat'], test_ac_adaptive.get_estimated()['reject_null']

Permutations:   0%|          | 0/50 [00:00<?, ?permutation/s]

(1.0, -0.00044244655245646327, False)

In [24]:
test_ac_adaptive = MultivariateACTest(k=55, 
                                      solver=ASE, 
                                      adaptive_m=True,
                                      adaptive_exponents=(0.1, 0.3),
                                      random_state=1, 
                                      verbose=True, 
                                      permutation_type='covariate', 
                                      n_jobs=-1,
                                      use_permutation_coeff=False,
                                      npermutations=50)

test_ac_adaptive.fit(data_with_outcome)

test_ac_adaptive.get_estimated()['p-value'], test_ac_adaptive.get_estimated()['test_stat'], test_ac_adaptive.get_estimated()['reject_null']

Permutations:   0%|          | 0/50 [00:00<?, ?permutation/s]

Process SpawnPoolWorker-59:
Process SpawnPoolWorker-55:
Traceback (most recent call last):
Traceback (most recent call last):
Process SpawnPoolWorker-61:
  File "/Users/lrcosta/opt/anaconda3/envs/network_independence/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/Users/lrcosta/opt/anaconda3/envs/network_independence/lib/python3.12/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/Users/lrcosta/opt/anaconda3/envs/network_independence/lib/python3.12/multiprocessing/pool.py", line 125, in worker
    result = (True, func(*args, **kwds))
                    ^^^^^^^^^^^^^^^^^^^
  File "/Users/lrcosta/umich/testing-network-independence/src/methods/_base_class.py", line 19, in _run_permutation_worker
    return _PERMUTATION_WORKER_METHOD._evaluate_permutation(task)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/lrcosta/umich/testing-network-independence/src/methods/

KeyboardInterrupt: 

In [53]:
3000/60

50.0